In [1]:
import os

In [2]:
%pwd

'c:\\Projects\\Text-Summarizer-project_01\\Text-Summarizer-project\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Projects\\Text-Summarizer-project_01\\Text-Summarizer-project'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    data_path: Path
    model_path: Path
    tokenizer_path: Path
    metric_file_name: Path

In [6]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            model_path = config.model_path,
            tokenizer_path = config.tokenizer_path,
            metric_file_name = config.metric_file_name
           
        )

        return model_evaluation_config


In [8]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset, load_from_disk
import torch
import pandas as pd
from tqdm import tqdm
from rouge_score import rouge_scorer

c:\Projects\Text-Summarizer-project_01\Text-Summarizer-project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
from rouge_score import rouge_scorer

rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

[2026-08-08 14:27:07,892: INFO: rouge_scorer: Using default tokenizer.]


In [10]:
class ModelEvaluation:

    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def generate_batch_sized_chunks(self, list_of_elements, batch_size):
        """Split dataset into smaller batches."""
        for i in range(0, len(list_of_elements), batch_size):
            yield list_of_elements[i:i + batch_size]

    def calculate_metric_on_test_ds(
        self,
        dataset,
        metric,
        model,
        tokenizer,
        batch_size=16,
        device="cuda" if torch.cuda.is_available() else "cpu",
        column_text="article",
        column_summary="highlights"
    ):

        article_batches = list(
            self.generate_batch_sized_chunks(
                dataset[column_text],
                batch_size
            )
        )

        target_batches = list(
            self.generate_batch_sized_chunks(
                dataset[column_summary],
                batch_size
            )
        )

        all_scores = {
            "rouge1": [],
            "rouge2": [],
            "rougeL": []
        }

        for article_batch, target_batch in tqdm(
            zip(article_batches, target_batches),
            total=len(article_batches)
        ):

            inputs = tokenizer(
                article_batch,
                max_length=1024,
                truncation=True,
                padding="max_length",
                return_tensors="pt"
            )

            summaries = model.generate(
                input_ids=inputs["input_ids"].to(device),
                attention_mask=inputs["attention_mask"].to(device),
                length_penalty=0.8,
                num_beams=8,
                max_length=128
            )

            decoded_summaries = [
                tokenizer.decode(
                    s,
                    skip_special_tokens=True,
                    clean_up_tokenization_spaces=True
                )
                for s in summaries
            ]

            for prediction, reference in zip(
                decoded_summaries,
                target_batch
            ):

                result = metric.score(
                    reference,
                    prediction
                )

                all_scores["rouge1"].append(
                    result["rouge1"].fmeasure
                )

                all_scores["rouge2"].append(
                    result["rouge2"].fmeasure
                )

                all_scores["rougeL"].append(
                    result["rougeL"].fmeasure
                )

        final_score = {
            "rouge1": sum(all_scores["rouge1"]) / len(all_scores["rouge1"]),
            "rouge2": sum(all_scores["rouge2"]) / len(all_scores["rouge2"]),
            "rougeL": sum(all_scores["rougeL"]) / len(all_scores["rougeL"])
        }

        return final_score

    def evaluate(self):

        device = "cuda" if torch.cuda.is_available() else "cpu"

        tokenizer = AutoTokenizer.from_pretrained(
            self.config.tokenizer_path
        )

        model_pegasus = AutoModelForSeq2SeqLM.from_pretrained(
            self.config.model_path
        ).to(device)

        dataset_samsum_pt = load_from_disk(
            self.config.data_path
        )

        rouge_metric = rouge_scorer.RougeScorer(
            ["rouge1", "rouge2", "rougeL"],
            use_stemmer=True
        )

        score = self.calculate_metric_on_test_ds(
            dataset_samsum_pt["test"][0:10],
            rouge_metric,
            model_pegasus,
            tokenizer,
            batch_size=2,
            column_text="dialogue",
            column_summary="summary"
        )

        rouge_dict = {
            "rouge1": score["rouge1"],
            "rouge2": score["rouge2"],
            "rougeL": score["rougeL"]
        }

        df = pd.DataFrame(
            rouge_dict,
            index=["pegasus"]
        )

        df.to_csv(
            self.config.metric_file_name,
            index=False
        )

        return df

In [11]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.evaluate()
except Exception as e:
    raise e

[2026-08-08 14:27:17,980: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-08-08 14:27:17,989: INFO: common: yaml file: params.yaml loaded successfully]
[2026-08-08 14:27:17,991: INFO: common: created directory at: artifacts]
[2026-08-08 14:27:17,993: INFO: common: created directory at: artifacts/model_evaluation]


Loading weights: 100%|██████████| 680/680 [00:00<00:00, 8276.61it/s]


[2026-08-08 14:27:21,572: INFO: rouge_scorer: Using default tokenizer.]


100%|██████████| 5/5 [03:08<00:00, 37.71s/it]
